In [1]:
import pandas as pd
from typing import List, Dict, Any, Union
#from statsbombpy import sb
import numpy as np
import json
import re
import psycopg 
import uuid

#N(reaS^!.sTijg7

In [2]:
database = 'localhost'
port = 5432
db_name = 'football_db'
password = 'Juanesg22'

def select_from_db(host,port,dbname,password,query):
    test = []
    with psycopg.connect(f"host={host} port={port} dbname={dbname} user=postgres password={password}") as conn:
                with conn.cursor() as curs:
                    curs.execute(query)
                    result_rows = curs.fetchall()
                    for row in result_rows:
                        test.append(row)
    return test

# insert data into the database
def insert_to_db(host,port,dbname,password,query,tuple_list):
    test = []
    
    with psycopg.connect(f"host={host} port={port} dbname={dbname} user=postgres password={password}") as conn:
                with conn.cursor() as curs:
                    curs.executemany(query,tuple_list)
                    conn.commit()
    return test


def extract_matchweek_number(matchweek_str):
    """
    Extracts the numeric part from a matchweek string.
    Examples:
        "JORNADA 38" -> 38
        "MATCHDAY 12" -> 12
        "GW5" -> 5
        "38" -> 38
    Returns integer or None if not found.
    """
    if pd.isna(matchweek_str):
        return None
    match = re.search(r'\d+', str(matchweek_str))
    return int(match.group()) if match else None


def extract_unique_players(df):
    """
    Extract unique player names from the columns: player, assist, player_in, player_out.
    Removes values containing special characters like ()/? and None values.
    Returns a sorted list of unique player names.
    """
    import re
    player_cols = ['player', 'assist', 'player_in', 'player_out']
    unique_players = set()
    for col in player_cols:
        if col in df.columns:
            unique_players.update(df[col].dropna().unique())
    # Remove entries with special characters and None
    filtered_players = [
        p for p in unique_players
        if isinstance(p, str) and not re.search(r"[()/\?]", p)
    ]
    player_uuid_list = [(p, uuid.uuid4()) for p in sorted(filtered_players)]
    return player_uuid_list

In [3]:
def extract_matches_to_dataframe(match_data_list: List[List]) -> pd.DataFrame:
    """
    Extract match information from nested lists and convert to a pandas DataFrame.
    
    Expected structure for each match:
    [competition_info, datetime, season, teams_dict, home_events_list, away_events_list]
    
    Args:
        match_data_list: List of lists, where each inner list represents a match
    
    Returns:
        pandas.DataFrame with all match and event information
    """
    all_rows = []
    
    for match_idx, match in enumerate(match_data_list):
        try:
            # Extract basic match information
            competition = match[0] if len(match) > 0 else None
            datetime_str = match[1] if len(match) > 1 else None
            season = match[2] if len(match) > 2 else None
            teams = match[3] if len(match) > 3 and isinstance(match[3], dict) else {}
            
            # Extract team information
            home_team = teams.get('home', None)
            away_team = teams.get('away', None)
            
            # Process events (home and away)
            events_lists = match[4:] if len(match) > 4 else []
            
            # If no events, create one row with match info only
            if not any(events_lists):
                row = {
                    'match_id': match_idx,
                    'competition': competition,
                    'datetime': datetime_str,
                    'season': season,
                    'home_team': home_team,
                    'away_team': away_team,
                    'event_type': None,
                    'time': None,
                    'player': None,
                    'team_side': None,
                    'score': None,
                    'assist': None,
                    'player_in': None,
                    'player_out': None,
                    'reason': None,
                    'event_source': None
                }
                all_rows.append(row)
            else:
                # Process each events list (typically home and away)
                for list_idx, events_list in enumerate(events_lists):
                    if not isinstance(events_list, list):
                        continue
                        
                    team_side = 'home' if list_idx == 0 else 'away'
                    
                    # If events list is empty, create one row for this team
                    if not events_list:
                        row = {
                            'match_id': match_idx,
                            'competition': competition,
                            'datetime': datetime_str,
                            'season': season,
                            'home_team': home_team,
                            'away_team': away_team,
                            'event_type': None,
                            'time': None,
                            'player': None,
                            'team_side': team_side,
                            'score': None,
                            'assist': None,
                            'player_in': None,
                            'player_out': None,
                            'reason': None,
                            'event_source': f'list_{list_idx}'
                        }
                        all_rows.append(row)
                        continue
                    
                    # Process each event in the list
                    for event in events_list:
                        if not isinstance(event, dict):
                            continue
                            
                        # Create base row with match information
                        row = {
                            'match_id': match_idx,
                            'competition': competition,
                            'datetime': datetime_str,
                            'season': season,
                            'home_team': home_team,
                            'away_team': away_team,
                            'team_side': team_side,
                            'event_source': f'list_{list_idx}'
                        }
                        
                        # Extract event information with flexible key handling
                        row['event_type'] = event.get('type', None)
                        row['time'] = event.get('time', None)
                        row['player'] = event.get('player', None)
                        row['score'] = event.get('score', None)
                        row['assist'] = event.get('assist', None)
                        row['player_in'] = event.get('player_in', None)
                        row['player_out'] = event.get('player_out', None)
                        row['reason'] = event.get('reason', None)
                        
                        # Override team_side if specified in event
                        if 'team' in event:
                            row['team_side'] = event['team']
                        
                        # Add any additional keys that might exist
                        for key, value in event.items():
                            if key not in ['type', 'time', 'player', 'score', 'assist', 
                                         'player_in', 'player_out', 'reason', 'team']:
                                row[f'extra_{key}'] = value
                        
                        all_rows.append(row)
                        
        except Exception as e:
            print(f"Error processing match {match_idx}: {e}")
            continue
    
    # Convert to DataFrame
    df = pd.DataFrame(all_rows)
    
    # Reorder columns for better readability
    base_columns = ['match_id', 'competition', 'datetime', 'season', 'home_team', 'away_team', 
                   'team_side', 'event_source', 'event_type', 'time', 'player', 'score', 
                   'assist', 'player_in', 'player_out', 'reason']
    
    # Add any extra columns that were found
    extra_columns = [col for col in df.columns if col.startswith('extra_')]
    final_columns = base_columns + extra_columns
    
    # Only include columns that exist in the DataFrame
    final_columns = [col for col in final_columns if col in df.columns]
    
    return df[final_columns]


def analyze_match_data(df: pd.DataFrame) -> Dict[str, Any]:
    """
    Provide a summary analysis of the extracted match data.
    
    Args:
        df: DataFrame returned by extract_matches_to_dataframe
    
    Returns:
        Dictionary with analysis summary
    """
    analysis = {
        'total_matches': df['match_id'].nunique(),
        'total_events': len(df[df['event_type'].notna()]),
        'competitions': df['competition'].unique().tolist(),
        'seasons': df['season'].unique().tolist(),
        'event_types': df['event_type'].value_counts().to_dict(),
        'teams': sorted(set(df['home_team'].dropna().tolist() + df['away_team'].dropna().tolist())),
        'date_range': {
            'earliest': df['datetime'].min(),
            'latest': df['datetime'].max()
        }
    }
    
    return analysis


def process_match_dataframe(df):
    """
    Process an existing DataFrame to:
    1. Split 'competition' column into 'competition' and 'matchweek'
    2. Add 'half_event' column based on 'time' column
    
    Args:
        df: pandas DataFrame with 'competition' and 'time' columns
    
    Returns:
        pandas DataFrame with processed columns
    """
    # Make a copy to avoid modifying the original
    df_processed = df.copy()
    
    # Split competition column
    df_processed = split_competition_column(df_processed)
    
    # Add half_event column
    df_processed = add_half_event_column(df_processed)
    
    return df_processed

def split_competition_column(df):
    """
    Split the 'competition' column into 'competition' and 'matchweek' columns.
    """
    df = df.copy()
    
    # Initialize new columns
    df['matchweek'] = None
    
    # Function to split individual competition strings
    def split_single_competition(comp_str):
        if pd.isna(comp_str) or not isinstance(comp_str, str):
            return comp_str, None
        
        # Common patterns for matchweek information
        patterns = [
            r'^(.+?)\s*-\s*(JORNADA\s+\d+)$',   # "PREMIER LEAGUE - JORNADA 38"
            r'^(.+?)\s*-\s*(MATCHDAY\s+\d+)$',  # "PREMIER LEAGUE - MATCHDAY 38"
            r'^(.+?)\s*-\s*(GAMEWEEK\s+\d+)$',  # "PREMIER LEAGUE - GAMEWEEK 38"
            r'^(.+?)\s*-\s*(WEEK\s+\d+)$',      # "PREMIER LEAGUE - WEEK 38"
            r'^(.+?)\s*-\s*(MD\s*\d+)$',        # "PREMIER LEAGUE - MD38"
            r'^(.+?)\s*-\s*(GW\s*\d+)$',        # "PREMIER LEAGUE - GW38"
            r'^(.+?)\s*-\s*(ROUND\s+\d+)$',     # "PREMIER LEAGUE - ROUND 38"
            r'^(.+?)\s*-\s*(\d+)$',             # "PREMIER LEAGUE - 38"
        ]
        
        for pattern in patterns:
            match = re.match(pattern, comp_str.strip(), re.IGNORECASE)
            if match:
                competition_name = match.group(1).strip()
                matchweek = match.group(2).strip()
                return competition_name, matchweek
        
        # If no pattern matches, return the full string as competition name
        return comp_str.strip(), None
    
    # Apply the splitting function
    split_results = df['competition'].apply(split_single_competition)
    
    # Update the columns
    df['competition'] = [result[0] for result in split_results]
    df['matchweek'] = [result[1] for result in split_results]
    
    return df

def add_half_event_column(df):
    """
    Add 'half_event' column based on the 'time' column.
    """
    df = df.copy()
    
    def determine_half(time_str):
        """
        Determine if an event occurred in the first or second half.
        """
        if pd.isna(time_str) or not isinstance(time_str, str):
            return None
        
        # Extract the base minute from formats like "74'", "45+2'", "90+1'"
        match = re.match(r'(\d+)', str(time_str).strip())
        if not match:
            return None
        
        try:
            minute = int(match.group(1))
            
            # Football halves: 1-45 minutes = First Half, 46+ minutes = Second Half
            if 1 <= minute <= 45:
                return 'First Half'
            elif minute >= 46:
                return 'Second Half'
            else:
                return None
        except ValueError:
            return None
    
    # Add the half_event column
    df['half_event'] = df['time'].apply(determine_half)
    
    return df

def reorder_columns(df):
    """
    Reorder columns for better readability, putting new columns in logical positions.
    """
    # Define preferred column order
    preferred_order = [
        'match_id', 'competition', 'matchweek', 'datetime', 'season', 
        'home_team', 'away_team', 'team_side', 'event_source', 
        'event_type', 'time', 'half_event', 'player', 'score', 
        'assist', 'player_in', 'player_out', 'reason'
    ]
    
    # Get existing columns
    existing_cols = df.columns.tolist()
    
    # Start with preferred columns that exist
    final_order = [col for col in preferred_order if col in existing_cols]
    
    # Add any remaining columns that weren't in the preferred list
    remaining_cols = [col for col in existing_cols if col not in final_order]
    final_order.extend(remaining_cols)
    
    return df[final_order]

In [4]:
with open(f'premier_league_data.json', 'r', encoding='utf-8') as f:
        read_data = json.load(f)

In [5]:
pd.DataFrame(read_data)[2]

0       2024/2025
1       2024/2025
2       2024/2025
3       2024/2025
4       2024/2025
          ...    
7215    2006/2007
7216    2006/2007
7217    2006/2007
7218    2006/2007
7219    2006/2007
Name: 2, Length: 7220, dtype: object

In [6]:
# for i in range(1,12):
#     print(i)
with open(f'premier_league_data.json', 'r', encoding='utf-8') as f:
        read_data = json.load(f)

df = extract_matches_to_dataframe(read_data)

df2 = process_match_dataframe(df)

goals_stats_teams = []

unique_seasons = list(df2['season'].unique())

for individual_season in unique_seasons:
    unique_teams = list(df2[df2['season'] == individual_season]['home_team'].unique())
    for individual_team in unique_teams:

        temporal_dict = {}

        home_goals_per_season = len(df2[(df2['home_team'] == individual_team) & (df2['event_type'] == 'goal') & (df2['team_side'] == 'home') & (df2['season'] == individual_season)])
        away_goals_per_season = len(df2[(df2['away_team'] == individual_team) & (df2['event_type'] == 'goal') & (df2['team_side'] == 'away') & (df2['season'] == individual_season)])

        average_goals_per_season = (home_goals_per_season + away_goals_per_season)/((len(unique_teams)-1)*2)

        # first half home season goals scored
        first_half_home_season_goals_scored = len(df2[(df2['home_team'] == individual_team) & (df2['event_type'] == 'goal') & (df2['team_side'] == 'home') & (df2['season'] == individual_season) & (df2['half_event'] == 'First Half')])
        # second half home season goals scored
        second_half_home_season_goals_scored = len(df2[(df2['home_team'] == individual_team) & (df2['event_type'] == 'goal') & (df2['team_side'] == 'home') & (df2['season'] == individual_season) & (df2['half_event'] == 'Second Half')])
        # first half away season goals scored
        first_half_away_season_goals_scored = len(df2[(df2['away_team'] == individual_team) & (df2['event_type'] == 'goal') & (df2['team_side'] == 'away') & (df2['season'] == individual_season) & (df2['half_event'] == 'First Half')])
        # first half away season goals scored
        second_half_away_season_goals_scored = len(df2[(df2['away_team'] == individual_team) & (df2['event_type'] == 'goal') & (df2['team_side'] == 'away') & (df2['season'] == individual_season) & (df2['half_event'] == 'Second Half')])
        # first half season goals average scored
        average_first_half_goals_scored = (first_half_home_season_goals_scored + first_half_away_season_goals_scored)/((len(unique_teams)-1)*2)
        # second half season goals average scored
        average_second_half_goals_scored = (second_half_away_season_goals_scored + second_half_home_season_goals_scored)/((len(unique_teams)-1)*2)

        average_home_game_first_half_goals_scored = first_half_home_season_goals_scored/(len(unique_teams)-1)

        average_home_game_second_half_goals_scored = second_half_home_season_goals_scored/(len(unique_teams)-1)

        average_away_game_first_half_goals_scored = first_half_away_season_goals_scored/(len(unique_teams)-1)

        average_away_game_second_half_goals_scored = second_half_away_season_goals_scored/(len(unique_teams)-1)


        # ...existing code...

# Goals conceded by individual_team (as home and away)
        home_goals_conceded_per_season = len(df2[
        
        (df2['away_team'] == individual_team) & 
        (df2['event_type'] == 'goal') & 
        (df2['team_side'] == 'home') & 
        (df2['season'] == individual_season)
        ])
        away_goals_conceded_per_season = len(df2[
        (df2['home_team'] == individual_team) & 
        (df2['event_type'] == 'goal') & 
        (df2['team_side'] == 'away') & 
        (df2['season'] == individual_season)
        ])

        # First/Second half conceded
        first_half_home_goals_conceded = len(df2[
        (df2['away_team'] == individual_team) & 
        (df2['event_type'] == 'goal') & 
        (df2['team_side'] == 'home') & 
        (df2['season'] == individual_season) & 
        (df2['half_event'] == 'First Half')
        ])
        second_half_home_goals_conceded = len(df2[
        (df2['away_team'] == individual_team) & 
        (df2['event_type'] == 'goal') & 
        (df2['team_side'] == 'home') & 
        (df2['season'] == individual_season) & 
        (df2['half_event'] == 'Second Half')
        ])
        first_half_away_goals_conceded = len(df2[
        (df2['home_team'] == individual_team) & 
        (df2['event_type'] == 'goal') & 
        (df2['team_side'] == 'away') & 
        (df2['season'] == individual_season) & 
        (df2['half_event'] == 'First Half')
        ])
        second_half_away_goals_conceded = len(df2[
        (df2['home_team'] == individual_team) & 
        (df2['event_type'] == 'goal') & 
        (df2['team_side'] == 'away') & 
        (df2['season'] == individual_season) & 
        (df2['half_event'] == 'Second Half')
        ])

        # Averages (adjust denominator as needed)
        average_home_game_goals_conceded = home_goals_conceded_per_season / (len(unique_teams)-1)
        average_away_game_goals_conceded = away_goals_conceded_per_season / (len(unique_teams)-1)
        average_first_half_home_goals_conceded = first_half_home_goals_conceded / (len(unique_teams)-1)
        average_second_half_home_goals_conceded = second_half_home_goals_conceded / (len(unique_teams)-1)
        average_first_half_away_goals_conceded = first_half_away_goals_conceded / (len(unique_teams)-1)
        average_second_half_away_goals_conceded = second_half_away_goals_conceded / (len(unique_teams)-1)
        # ...existing code...



                # print(f'season: {individual_season} , team: {individual_team}: home goals: {home_goals_per_season}, away goals: {away_goals_per_season}, average goals: {average_goals_per_season}')
                # print('==============================')
                # print(f'first half all season goals scored: {first_half_home_season_goals_scored + first_half_away_season_goals_scored} second half all season goals scored: {second_half_away_season_goals_scored + second_half_home_season_goals_scored}')
                # print('==============================')
                # print(f'average goals first half: {average_first_half_goals_scored} average goals second half: {average_second_half_goals_scored}')
                # print('==============================')
                # print(f'home game first half goals scored: {first_half_home_season_goals_scored} home game second half goals scored {second_half_home_season_goals_scored}')
                # print('==============================')
                # print(f'average home game first half goals: {average_home_game_first_half_goals_scored} average home game second half goals: {average_home_game_second_half_goals_scored}')
                # print('==============================')
                # print(f'away game first half goals scored:{first_half_away_season_goals_scored}  away game second half goals scored:{second_half_away_season_goals_scored}')
                # print('==============================')
                # print(f'average away game first half goals: {average_away_game_first_half_goals_scored} average away game second half goals: {average_away_game_second_half_goals_scored}')

        temporal_dict.update({
    'season': individual_season,
    'team': individual_team,
    'home_goals_season': home_goals_per_season,
    'away_goals_season': away_goals_per_season,
    'average_goals_season': average_goals_per_season,
    'first_half_season_goals_scored': first_half_home_season_goals_scored + first_half_away_season_goals_scored,
    'second_half_season_goals_scored': second_half_away_season_goals_scored + second_half_home_season_goals_scored,
    'average_season_first_half_goals_scored': average_first_half_goals_scored,
    'average_season_second_half_goals_scored': average_second_half_goals_scored,
    'first_half_home_season_goals_scored': first_half_home_season_goals_scored,
    'second_half_home_season_goals_scored': second_half_home_season_goals_scored,
    'average_home_game_first_half_goals_scored': average_home_game_first_half_goals_scored,
    'average_home_game_second_half_goals_scored': average_home_game_second_half_goals_scored,
    'first_half_away_season_goals_scored': first_half_away_season_goals_scored,
    'second_half_away_season_goals_scored': second_half_away_season_goals_scored,
    'average_away_game_first_half_goals_scored': average_away_game_first_half_goals_scored,
    'average_away_game_second_half_goals_scored': average_away_game_second_half_goals_scored,

    # NEW: Goals conceded stats
    'home_goals_conceded_season': home_goals_conceded_per_season,
    'away_goals_conceded_season': away_goals_conceded_per_season,
    'first_half_home_goals_conceded': first_half_home_goals_conceded,
    'second_half_home_goals_conceded': second_half_home_goals_conceded,
    'first_half_away_goals_conceded': first_half_away_goals_conceded,
    'second_half_away_goals_conceded': second_half_away_goals_conceded,
    'average_home_game_goals_conceded': average_home_game_goals_conceded,
    'average_away_game_goals_conceded': average_away_game_goals_conceded,
    'average_first_half_home_goals_conceded': average_first_half_home_goals_conceded,
    'average_second_half_home_goals_conceded': average_second_half_home_goals_conceded,
    'average_first_half_away_goals_conceded': average_first_half_away_goals_conceded,
    'average_second_half_away_goals_conceded': average_second_half_away_goals_conceded
})
        # print(temporal_dict)
        goals_stats_teams.append(temporal_dict)
with open(f'goal_stats_league{0}.json', 'w', encoding='utf-8') as f:
        json.dump(temporal_dict, f, ensure_ascii=False, indent=4)    


 #   i += 1

In [7]:
df2.head(1)

,match_id,competition,datetime,season,home_team,away_team,team_side,event_source,event_type,time,player,score,assist,player_in,player_out,reason,matchweek,half_event
0,0,PREMIER LEAGUE,25.05.2025 10:00,2024/2025,Bournemouth,Leicester,home,list_0,substitution,63',Jebbison D.,None,None,Jebbison D.,Brooks D.,None,JORNADA 38,Second Half


In [8]:
season_names = df2['season'].unique().tolist()
competition_name = df2['competition'].unique().tolist()
premier_league_teams = df2['home_team'].unique().tolist()

In [9]:
# Given a list like ['2024/2025', '2023/2024', ...]
season_names_df = pd.DataFrame(season_names, columns=['season'])

# Extract start and end years from the season string
season_names_df['start_year'] = season_names_df['season'].str.split('/').str[0]
season_names_df['end_year'] = season_names_df['season'].str.split('/').str[1]

# Create open_date (August 1st of start year) and close_date (July 31st of end year)
season_names_df['open_date'] = season_names_df['start_year'].apply(lambda x: f"{x}-08-01")
season_names_df['close_date'] = season_names_df['end_year'].apply(lambda x: f"{x}-07-31")

# Drop helper columns if not needed
season_names_df = season_names_df.drop(['start_year', 'end_year'], axis=1)

season_names_df['created_at'] = pd.Timestamp.now()
season_names_df['pk_season_id'] = [uuid.uuid4() for _ in range(len(season_names_df))]
season_names_df['is_active'] = False
season_names_df.head(1)

,season,open_date,close_date,created_at,pk_season_id,is_active
0,2024/2025,2024-08-01,2025-07-31,2025-07-28 22:07:38.675278,46b947b8-e1eb-4dea-87bd-af67b71ca41d,False


In [10]:
insert_to_db(database, port, db_name, password,
             """INSERT INTO seasons (season_name, start_date, end_date, created_at, pk_season_id, is_active)
                VALUES (%s, %s, %s, %s, %s, %s)""",
             list(season_names_df.itertuples(index=False, name=None)))

[]

In [11]:
pk_uk_id = select_from_db(database, port, db_name, password,
               """
    SELECT pk_country_id
    FROM countries
    WHERE country_name = 'United Kingdom';
    """
)[0][0]

In [12]:
competition_name_df = pd.DataFrame(competition_name, columns=['competition_name'])
competition_name_df['created_at'] = pd.Timestamp.now()
competition_name_df['pk_competition_id'] = [uuid.uuid4() for _ in range(len(competition_name_df))]
competition_name_df['season_format'] = 'anual'
competition_name_df['competition_type'] = 'league'
competition_name_df['fk_country_id'] = pk_uk_id
competition_name_df

,competition_name,created_at,pk_competition_id,season_format,competition_type,fk_country_id
0,PREMIER LEAGUE,2025-07-28 22:07:38.898386,d9afe753-0ca0-44ac-ae5b-d88d05e9cfb5,anual,league,c2e3fa41-fe02-40e7-9f37-c43520a683c6


In [13]:
insert_to_db(database, port, db_name, password,
             """ INSERT INTO competitions (competition_name, created_at, pk_competition_id, season_format,competition_type, fk_country_id)
                VALUES (%s, %s, %s, %s, %s,%s)""",
                list(competition_name_df.itertuples(index=False, name=None)))

[]

In [14]:
competitions_seasons = season_names_df[['pk_season_id','open_date', 'close_date']]
competitions_seasons['created_at'] = pd.Timestamp.now()
competitions_seasons['pk_competition_season_id'] = [uuid.uuid4() for _ in range(len(competitions_seasons))]
competitions_seasons['fk_competition_id'] = competition_name_df['pk_competition_id'].iloc[0]  # Assuming one competition for all seasons
competitions_seasons['total_matchweeks'] = 38  # Assuming 38 matchweeks for Premier League

insert_to_db(database, port, db_name, password,
             """INSERT INTO competition_seasons (fk_season_id, start_date, end_date, created_at,pk_competition_season_id,fk_competition_id, total_matchweeks)
                VALUES (%s, %s, %s, %s, %s, %s, %s)""",
             list(competitions_seasons.itertuples(index=False, name=None)))

C:\Users\Janus\AppData\Local\Temp\ipykernel_9772\1560374200.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  competitions_seasons['created_at'] = pd.Timestamp.now()
C:\Users\Janus\AppData\Local\Temp\ipykernel_9772\1560374200.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  competitions_seasons['pk_competition_season_id'] = [uuid.uuid4() for _ in range(len(competitions_seasons))]


[]

In [15]:
premier_league_teams_df = pd.DataFrame(premier_league_teams, columns=['team_name'])
premier_league_teams_df['created_at'] = pd.Timestamp.now()
premier_league_teams_df['pk_team_id'] = [uuid.uuid4() for _ in range(len(premier_league_teams_df))]
premier_league_teams_df['fk_country_id'] = pk_uk_id

insert_to_db(database, port, db_name, password,
             """INSERT INTO teams (team_name, created_at, pk_team_id, fk_country_id)
                VALUES (%s, %s, %s, %s)""",
                list(premier_league_teams_df.itertuples(index=False, name=None)))


[]

In [16]:
# Generate a unique UUID for each match_id in df2
df2['pk_match_id'] = df2['match_id'].map(lambda x: uuid.uuid4())

In [17]:
matches = df2[['match_id','home_team','away_team','season','team_side','event_type','matchweek']]

In [18]:
# Count home and away goals per match_id, filling missing matches with zeros

# Filter only goal events
goals = matches[matches['event_type'] == 'goal']

# Count goals by match_id and team_side
goal_counts = (
    goals.groupby(['match_id', 'team_side'])
    .size()
    .unstack(fill_value=0)
    .rename(columns={'home': 'home_goals', 'away': 'away_goals'})
)

# Ensure all match_ids are present, fill missing with zeros
all_matches = matches[['match_id', 'home_team', 'away_team', 'season', 'matchweek']].drop_duplicates().set_index('match_id')
goals_summary = all_matches.join(goal_counts, how='left').fillna({'home_goals': 0, 'away_goals': 0})

# Convert goal columns to integer
goals_summary['home_goals'] = goals_summary['home_goals'].astype(int)
goals_summary['away_goals'] = goals_summary['away_goals'].astype(int)

goals_summary = goals_summary.reset_index()

match_id_to_uuid = {mid: uuid.uuid4() for mid in matches['match_id'].unique()}

# Assign the same uuid4 to all rows with the same match_id
goals_summary['pk_match_id'] = goals_summary['match_id'].map(match_id_to_uuid)

pks_competitions_seasons_df = pd.DataFrame(select_from_db(database, port, db_name, password, 
               '''
              select cs.pk_competition_season_id, c.competition_name, s.season_name
                from competition_seasons cs
                join competitions c on cs.fk_competition_id = c.pk_competition_id
                join public.seasons s on cs.fk_season_id = s.pk_season_id
                '''), columns=['pk_competition_season_id', 'competition_name', 'season_name'])

goals_summary['fk_home_team_id'] = goals_summary['home_team'].map(premier_league_teams_df.set_index('team_name')['pk_team_id'])
goals_summary['fk_away_team_id'] = goals_summary['away_team'].map(premier_league_teams_df.set_index('team_name')['pk_team_id'])

# Add datetime column from df2 to goals_summary based on match_id
goals_summary = goals_summary.merge(
    df2[['match_id', 'datetime']].drop_duplicates(),
    on='match_id',
    how='left'
)

goals_summary['fk_competition_season_id'] = goals_summary['season'].map(
    pks_competitions_seasons_df.set_index('season_name')['pk_competition_season_id'])

goals_summary_v2 = goals_summary.copy()

goals_summary.drop(columns=['match_id','season','home_team','away_team'], inplace=True)
goals_summary['matchweek_number'] = goals_summary['matchweek'].apply(extract_matchweek_number)
goals_summary['match_status'] = 'completed'  # Assuming all matches are completed for this dataset

goals_summary


insert_to_db(database, port, db_name, password,
             """INSERT INTO matches (matchweek,home_score, away_score,pk_match_id, fk_home_team_id, fk_away_team_id,kick_off,fk_competition_season_id, matchweek_number, match_status)
                VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s)""",
             list(goals_summary.itertuples(index=False, name=None)))

[]

In [19]:
# Example usage:
unique_players = extract_unique_players(df2)

players_df = pd.DataFrame(unique_players, columns=['player_name' , 'pk_player_id'])

players_df['created_at'] = pd.Timestamp.now()
players_df['second_name'] = ''

insert_to_db(database, port, db_name, password,
             """INSERT INTO players (first_name,  pk_player_id,created_at, last_name)
                VALUES (%s, %s, %s, %s)""",
             list(players_df.itertuples(index=False, name=None)))

[]

In [20]:
# Replace values in the 'event_type' column as specified
df2['event_type'] = df2['event_type'].replace({
    'goal': 'Goal',
    'substitution': 'Substitution',
    'yellow_card': 'Yellow Card',
    'red_card': 'Red Card'
})

action_type_list = select_from_db(database, port, db_name, password,
               """ 
                  SELECT pk_action_type_id, action_name
                  from action_types""")

action_type_df = pd.DataFrame(action_type_list, columns=['pk_action_type_id', 'action_name'])


In [21]:
match_events = df2[['match_id','home_team','away_team','team_side','event_type','time','player','assist','player_in','player_out','reason','half_event']]

In [22]:
match_events_v2 = pd.merge(match_events,goals_summary_v2[['match_id','pk_match_id']],how='left', on='match_id')

In [23]:
match_events_v2.head(1)

,match_id,home_team,away_team,team_side,event_type,time,player,assist,player_in,player_out,reason,half_event,pk_match_id
0,0,Bournemouth,Leicester,home,Substitution,63',Jebbison D.,None,Jebbison D.,Brooks D.,None,Second Half,edb4b1ee-0dcf-43f9-8f8e-5ff9a0b13ab3


In [24]:
# Create a column to use for joining based on team_side
match_events_v2['team_for_join'] = match_events_v2.apply(
    lambda row: row['home_team'] if row['team_side'] == 'home' else row['away_team'], axis=1
)

# Merge with premier_league_teams_df on the computed team_for_join and team_name
match_events_v3= match_events_v2.merge(
    premier_league_teams_df,
    left_on='team_for_join',
    right_on='team_name',
    how='left',
    suffixes=('', '_team')
)

match_events_v4 = match_events_v3[['event_type','time','player','assist','player_in','player_out','reason','pk_match_id','pk_team_id','half_event']]

In [25]:
match_events_v4 = match_events_v4[match_events_v4['event_type'].notna()]

match_events_v5 = pd.merge(match_events_v4,action_type_df,how='left', left_on='event_type', right_on='action_name').drop(columns=['event_type', 'action_name'])

match_events_v5

,time,player,assist,player_in,player_out,reason,pk_match_id,pk_team_id,half_event,pk_action_type_id
0,63',Jebbison D.,None,Jebbison D.,Brooks D.,None,edb4b1ee-0dcf-43f9-8f8e-5ff9a0b13ab3,3c2fb529-3746-45a9-88ba-abc6b0587360,Second Half,f43051e4-b159-40ac-9c32-f37723c77afc
1,74',Semenyo A.,Zabarnyi I.,None,None,None,edb4b1ee-0dcf-43f9-8f8e-5ff9a0b13ab3,3c2fb529-3746-45a9-88ba-abc6b0587360,Second Half,f49dcb4f-fe83-4b62-ac59-1abc440a1f64
2,78',Huijsen D.,None,Huijsen D.,Senesi M.,None,edb4b1ee-0dcf-43f9-8f8e-5ff9a0b13ab3,3c2fb529-3746-45a9-88ba-abc6b0587360,Second Half,f43051e4-b159-40ac-9c32-f37723c77afc
3,88',Semenyo A.,Huijsen D.,None,None,None,edb4b1ee-0dcf-43f9-8f8e-5ff9a0b13ab3,3c2fb529-3746-45a9-88ba-abc6b0587360,Second Half,f49dcb4f-fe83-4b62-ac59-1abc440a1f64
4,90+1',Scott A.,None,Scott A.,Evanilson,None,edb4b1ee-0dcf-43f9-8f8e-5ff9a0b13ab3,3c2fb529-3746-45a9-88ba-abc6b0587360,Second Half,f43051e4-b159-40ac-9c32-f37723c77afc
...,...,...,...,...,...,...,...,...,...,...
86827,34',Agger D.,None,Agger D.,Carragher J.,None,4cdcbb1d-8d81-49ae-b379-4d297a33f7e7,3709f7ff-2650-4112-9acd-71bc1f7907ca,First Half,f43051e4-b159-40ac-9c32-f37723c77afc
86828,64',Sissoko M.,None,None,None,None,4cdcbb1d-8d81-49ae-b379-4d297a33f7e7,3709f7ff-2650-4112-9acd-71bc1f7907ca,Second Half,e3be0248-968b-4d7a-a9e8-a59047b06721
86829,70',Fowler R.,None,None,None,None,4cdcbb1d-8d81-49ae-b379-4d297a33f7e7,3709f7ff-2650-4112-9acd-71bc1f7907ca,Second Half,f49dcb4f-fe83-4b62-ac59-1abc440a1f64
86830,82',Kromkamp J.,None,None,None,None,4cdcbb1d-8d81-49ae-b379-4d297a33f7e7,3709f7ff-2650-4112-9acd-71bc1f7907ca,Second Half,e3be0248-968b-4d7a-a9e8-a59047b06721


In [26]:
# Join player column to get fk_player_id
match_events_v6 = match_events_v5.merge(
    players_df[['player_name', 'pk_player_id']],
    how='left',
    left_on='player',
    right_on='player_name'
).rename(columns={'pk_player_id': 'fk_player_id'}).drop(columns=['player_name'])

# Combine assist and player_out into a single column for join
match_events_v6['assist_or_out'] = match_events_v6['assist'].combine_first(match_events_v6['player_out'])

# Join assist_or_out column to get fk_assist_or_out_player_id
match_events_v6 = match_events_v6.merge(
    players_df[['player_name', 'pk_player_id']],
    how='left',
    left_on='assist_or_out',
    right_on='player_name'
).rename(columns={'pk_player_id': 'fk_secondary_player_id'}).drop(columns=['player_name','player','assist','player_in','player_out', 'assist_or_out'])

In [27]:
match_events_v6['fk_secondary_player_id'] = match_events_v6['fk_secondary_player_id'].where(
    match_events_v6['fk_secondary_player_id'].notna(), None
)

match_events_v6['fk_player_id'] = match_events_v6['fk_player_id'].where(
    match_events_v6['fk_player_id'].notna(), None
)

In [28]:
df2.head(1)

,match_id,competition,datetime,season,home_team,away_team,team_side,event_source,event_type,time,player,score,assist,player_in,player_out,reason,matchweek,half_event,pk_match_id
0,0,PREMIER LEAGUE,25.05.2025 10:00,2024/2025,Bournemouth,Leicester,home,list_0,Substitution,63',Jebbison D.,None,None,Jebbison D.,Brooks D.,None,JORNADA 38,Second Half,986c5427-a2c2-44c6-b7a6-1321d2cd3439


In [29]:
match_events_v6.head(1)

,time,reason,pk_match_id,pk_team_id,half_event,pk_action_type_id,fk_player_id,fk_secondary_player_id
0,63',None,edb4b1ee-0dcf-43f9-8f8e-5ff9a0b13ab3,3c2fb529-3746-45a9-88ba-abc6b0587360,Second Half,f43051e4-b159-40ac-9c32-f37723c77afc,1665e67d-2459-42ca-a159-0ee5f699b68c,4736f162-527f-4a6d-87df-bcca3130166f


In [30]:
# Extract main minute and additional time from 'time' column
def extract_minute_and_extra(time_str):
    if pd.isna(time_str):
        return (None, None)
    match = re.match(r"(\d+)(?:\+(\d+))?", str(time_str))
    if match:
        minute = int(match.group(1))
        extra = int(match.group(2)) if match.group(2) else None
        return (minute, extra)
    return (None, None)

match_events_v6[['minute', 'additional_time']] = match_events_v6['time'].apply(
    extract_minute_and_extra
).apply(pd.Series)

match_events_v6.drop(columns=['time'], inplace=True)

match_events_v6['additional_time'].fillna(0, inplace=True)
#match_events_v6['additional_time'] = match_events_v6['additional_time'].astype('Int64')



In [31]:
print("Minute max:", match_events_v6['minute'].max())
print("Minute min:", match_events_v6['minute'].min())
print("Additional time max:", match_events_v6['additional_time'].max())
print("Additional time min:", match_events_v6['additional_time'].min())

Minute max: 90.0
Minute min: 1.0
Additional time max: 20.0
Additional time min: 0.0


In [32]:
# Show rows where both 'minute' and 'additional_time' are None
missing_minute_rows = match_events_v6[
    match_events_v6['minute'].isna() & match_events_v6['additional_time'].isna()
]
missing_minute_rows

,reason,pk_match_id,pk_team_id,half_event,pk_action_type_id,fk_player_id,fk_secondary_player_id,minute,additional_time


In [33]:
import uuid

def convert_uuid_to_string(x):
    """Convert UUID objects to strings, handle None values"""
    if pd.isna(x) or x is None:
        return None
    elif isinstance(x, uuid.UUID):
        return str(x)
    elif isinstance(x, str):
        return x  # Already a string
    else:
        return str(x)  # Convert whatever it is to string

# Convert UUID columns to strings
match_events_v6['fk_player_id'] = match_events_v6['fk_player_id'].apply(convert_uuid_to_string)
match_events_v6['fk_secondary_player_id'] = match_events_v6['fk_secondary_player_id'].apply(convert_uuid_to_string)

# Also make sure your minute columns are properly converted
match_events_v6['minute'] = match_events_v6['minute'].apply(lambda x: int(x) if pd.notnull(x) else None)
match_events_v6['additional_time'] = match_events_v6['additional_time'].apply(lambda x: int(x) if pd.notnull(x) else None)

In [34]:
match_events_v6.dtypes

reason                     object
pk_match_id                object
pk_team_id                 object
half_event                 object
pk_action_type_id          object
fk_player_id               object
fk_secondary_player_id     object
minute                    float64
additional_time             int64
dtype: object

In [35]:
def df_to_python_tuples(df):
    """Convert DataFrame to list of tuples with pure Python types"""
    # Convert to records (list of dicts)
    records = df.to_dict('records')
    
    tuples_list = []
    for record in records:
        row = []
        for key, value in record.items():
            if pd.isna(value) or value is None:
                row.append(None)
            elif isinstance(value, (int, np.int64, np.int32, np.integer)):
                row.append(int(value))
            elif isinstance(value, (float, np.float64, np.float32, np.floating)):
                if key in ['minute', 'additional_time', 'event_minute']:
                    row.append(int(value))
                else:
                    row.append(float(value))
            else:
                row.append(str(value))
        tuples_list.append(tuple(row))
    
    return tuples_list

# Convert using this method
python_tuples = df_to_python_tuples(match_events_v6)

# Test one tuple
print("Sample converted tuple:")
for i, item in enumerate(python_tuples[0]):
    print(f"  {i}: {item} (type: {type(item)})")

# Try the insert
insert_to_db(database, port, db_name, password,
    """INSERT INTO match_events (event_description,fk_match_id, fk_team_id, half_period ,fk_action_type_id, fk_player_id, fk_secondary_player_id, event_minute, additional_time) 
    VALUES (%s,%s, %s, %s, %s, %s, %s, %s, %s)""",
    python_tuples)

Sample converted tuple:
  0: None (type: <class 'NoneType'>)
  1: edb4b1ee-0dcf-43f9-8f8e-5ff9a0b13ab3 (type: <class 'str'>)
  2: 3c2fb529-3746-45a9-88ba-abc6b0587360 (type: <class 'str'>)
  3: Second Half (type: <class 'str'>)
  4: f43051e4-b159-40ac-9c32-f37723c77afc (type: <class 'str'>)
  5: 1665e67d-2459-42ca-a159-0ee5f699b68c (type: <class 'str'>)
  6: 4736f162-527f-4a6d-87df-bcca3130166f (type: <class 'str'>)
  7: 63 (type: <class 'int'>)
  8: 0 (type: <class 'int'>)


error ignored terminating <psycopg.Pipeline [INERROR, pipeline=ON] (host=localhost user=postgres database=football_db) at 0x193cc02c950>: pipeline aborted


NotNullViolation: el valor nulo en la columna «half_period» de la relación «match_events» viola la restricción de no nulo
DETAIL:  La fila que falla contiene (9b9a02df-89bf-4a84-ad36-85f6f970c3ea, 9e9f0652-e785-4027-8742-88d380dd5bcb, f49dcb4f-fe83-4b62-ac59-1abc440a1f64, c8910d83-1058-4fb6-b888-f6c9066c2754, null, null, null, 0, null, null, null, null, 2025-07-28 22:07:50.319117).

In [241]:
match_events_v6_forced.dtypes

reason                    object
pk_match_id               object
pk_team_id                object
half_event                object
pk_action_type_id         object
fk_player_id              object
fk_secondary_player_id    object
minute                     Int64
additional_time            int64
dtype: object

In [237]:
match_events_v6['minute'] = match_events_v6['minute'].astype('Int64')
match_events_v6['additional_time'] = match_events_v6['additional_time'].apply(lambda x: int(x) if pd.notnull(x) else None)

insert_to_db(database, port, db_name, password,
             """INSERT INTO match_events (event_description,fk_match_id, fk_team_id, half_period  ,fk_action_type_id, fk_player_id, fk_secondary_player_id, event_minute, additional_time)
                VALUES (%s,%s, %s, %s, %s, %s, %s, %s, %s)""",
             list(match_events_v6.itertuples(index=False, name=None)))

ProgrammingError: cannot adapt type 'int64' using placeholder '%s' (format: AUTO)